# Industrial anomaly detection without training

MVTec AD is fifteen categories of manufactured parts. The training set contains **only
defect-free** examples; the test set contains both. The usual approach trains a model per
category.

This trains nothing. A frozen ImageNet ResNet-50 produces patch features, the features of
the defect-free training images go into a memory bank, and a test patch is scored by its
distance to the nearest normal patch. The only thing stored is what normal looks like.

Two things are measured that a single AUROC hides:

1. **Per-category AUROC**, because the fifteen categories are not one problem - textures
   and objects fail differently.
2. **A raw-pixel control** running the identical pipeline on downsampled pixels instead
   of network features. Without it, a good score could be the method rather than the
   representation, and there would be no way to tell.

## What is actually attached

In [ ]:
import os, glob, json, time
import numpy as np

ROOT = None
for cand in sorted(glob.glob("/kaggle/input/*")):
    # MVTec ships <category>/train/good/*.png - find whichever depth that sits at.
    hit = glob.glob(os.path.join(cand, "**", "train", "good"), recursive=True)
    if hit:
        ROOT = os.path.dirname(os.path.dirname(hit[0]))
        ROOT = os.path.dirname(ROOT) if os.path.basename(ROOT) != cand else ROOT
        ROOT = cand
        break
assert ROOT, "no <category>/train/good found under /kaggle/input"

CATS = sorted({p.split(os.sep)[-3] for p in
               glob.glob(os.path.join(ROOT, "**", "train", "good"), recursive=True)})
print("root:", ROOT)
print(f"{len(CATS)} categories: {CATS}")
assert len(CATS) >= 10, f"expected ~15 MVTec categories, found {len(CATS)}"

def cat_paths(c):
    base = glob.glob(os.path.join(ROOT, "**", c, "train", "good"), recursive=True)[0]
    cdir = os.path.dirname(os.path.dirname(base))
    train = sorted(glob.glob(os.path.join(cdir, "train", "good", "*.png")))
    test, labels = [], []
    for d in sorted(glob.glob(os.path.join(cdir, "test", "*"))):
        kind = os.path.basename(d)
        for f in sorted(glob.glob(os.path.join(d, "*.png"))):
            test.append(f); labels.append(0 if kind == "good" else 1)
    return train, test, np.array(labels)

tr, te, lb = cat_paths(CATS[0])
print(f"{CATS[0]}: {len(tr)} train(good), {len(te)} test, {int(lb.sum())} anomalous")
assert len(tr) > 20 and lb.sum() > 0 and (lb == 0).sum() > 0

## A frozen feature extractor

In [ ]:
import torch, torch.nn.functional as F
import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(max(1, (os.cpu_count() or 4)))
print("device:", DEV, "| threads:", torch.get_num_threads())

W = ResNet50_Weights.IMAGENET1K_V2
net = resnet50(weights=W).eval().to(DEV)
for p in net.parameters():
    p.requires_grad_(False)

# layer2 + layer3: mid-level texture and part structure. layer4 is too semantic for
# "this screw has a scratch", layer1 is close to raw edges.
feats = {}
net.layer2.register_forward_hook(lambda m, i, o: feats.__setitem__("l2", o))
net.layer3.register_forward_hook(lambda m, i, o: feats.__setitem__("l3", o))

SIZE = 256
tf = W.transforms()

def load(p):
    im = Image.open(p).convert("RGB").resize((SIZE, SIZE), Image.BILINEAR)
    return tf(im)

@torch.no_grad()
def patch_features(paths, bs=16):
    """(N*H*W, C) patch features and the (H, W) grid they came from."""
    out, grid = [], None
    for i in range(0, len(paths), bs):
        x = torch.stack([load(p) for p in paths[i:i + bs]]).to(DEV)
        net(x)
        a, b = feats["l2"], feats["l3"]
        b = F.interpolate(b, size=a.shape[-2:], mode="bilinear", align_corners=False)
        f = torch.cat([a, b], 1)
        f = F.avg_pool2d(f, 3, 1, 1)                       # local smoothing
        grid = f.shape[-2:]
        out.append(f.permute(0, 2, 3, 1).reshape(-1, f.shape[1]).cpu())
    return torch.cat(out), grid

print("feature extractor ready (frozen, nothing is trained)")

## Scoring, and the control

In [ ]:
from sklearn.metrics import roc_auc_score

def knn_scores(bank, q, k=1, chunk=4096):
    """Distance from each query patch to its k-th nearest bank patch."""
    d_out = []
    bank_n = (bank ** 2).sum(1)
    for i in range(0, len(q), chunk):
        qq = q[i:i + chunk]
        d = (qq ** 2).sum(1, keepdim=True) + bank_n[None, :] - 2 * qq @ bank.T
        d = d.clamp_min_(0)
        d_out.append(d.topk(k, dim=1, largest=False).values[:, -1].sqrt())
    return torch.cat(d_out)

def image_scores(bank, paths, grid_n, feat_fn, sub=None):
    q, grid = feat_fn(paths)
    s = knn_scores(bank, q)
    s = s.reshape(len(paths), -1)
    return s.max(1).values.numpy(), s.reshape(len(paths), *grid).numpy()

def raw_pixel_features(paths, bs=32, side=32):
    """The control: the same pipeline on downsampled raw pixels, no network."""
    out = []
    for p in paths:
        im = Image.open(p).convert("RGB").resize((side, side), Image.BILINEAR)
        a = torch.from_numpy(np.asarray(im, dtype="float32") / 255.0)
        out.append(a.reshape(-1, 3))
    return torch.cat(out), (side, side)

print("scoring defined: image score = max patch distance to the nearest normal patch")

## All fifteen categories

In [ ]:
MAX_BANK = 8000        # random subsample of normal patches; keeps memory flat
rng = np.random.default_rng(0)
rows, t0 = [], time.perf_counter()

for ci, c in enumerate(CATS):
    tr, te, lb = cat_paths(c)
    res = {"category": c, "n_train_good": len(tr), "n_test": len(te),
           "n_anomalous": int(lb.sum())}
    for name, fn in (("resnet50", patch_features), ("raw_pixels", raw_pixel_features)):
        bank, _ = fn(tr)
        if len(bank) > MAX_BANK:
            bank = bank[rng.choice(len(bank), MAX_BANK, replace=False)]
        q, grid = fn(te)
        s = knn_scores(bank, q).reshape(len(te), -1).max(1).values.numpy()
        res[f"auroc_{name}"] = float(roc_auc_score(lb, s))
    rows.append(res)
    print(f"  {c:14} resnet50 {res['auroc_resnet50']:.4f}   "
          f"raw_pixels {res['auroc_raw_pixels']:.4f}   "
          f"({len(tr)} good, {len(te)} test)")

assert len(rows) == len(CATS)
mean_f = float(np.mean([r["auroc_resnet50"] for r in rows]))
mean_p = float(np.mean([r["auroc_raw_pixels"] for r in rows]))
print(f"\nmean image AUROC - frozen features {mean_f:.4f} | raw pixels {mean_p:.4f}")
print(f"elapsed {(time.perf_counter()-t0)/60:.1f} min")

## Result

In [ ]:
BAR = 0.90       # published PatchCore reaches ~0.991; a plain kNN should clear 0.90
verdict = "PASS" if mean_f >= BAR else ("BELOW BAR" if mean_f >= 0.75 else "LIKELY BUG")
print(f"bar {BAR}: mean {mean_f:.4f} -> {verdict}")
print()
best = sorted(rows, key=lambda r: -r["auroc_resnet50"])[:3]
worst = sorted(rows, key=lambda r: r["auroc_resnet50"])[:3]
print("best :", ", ".join(f"{r['category']} {r['auroc_resnet50']:.4f}" for r in best))
print("worst:", ", ".join(f"{r['category']} {r['auroc_resnet50']:.4f}" for r in worst))

results = {
    "task": "MVTec AD image-level anomaly detection, no training",
    "method": "frozen ImageNet ResNet-50 layer2+layer3 patch features, "
              "memory bank of defect-free train patches, score = max 1-NN distance",
    "control": "identical pipeline on 32x32 raw pixels instead of network features",
    "bank_cap_patches": MAX_BANK,
    "device": DEV,
    "bar_mean_auroc": BAR,
    "mean_auroc_resnet50": mean_f,
    "mean_auroc_raw_pixels": mean_p,
    "feature_advantage": mean_f - mean_p,
    "per_category": rows,
}
with open("results.json", "w") as fh:
    json.dump(results, fh, indent=1)
print("\nwrote results.json")

## What this shows, and what it does not

Nothing was trained. The only fitted object is a memory bank of patch features from
defect-free images, so whatever the score turns out to be, it comes from an ImageNet
representation and a nearest-neighbour lookup.

The raw-pixel control is what makes that claim checkable. Both columns use the same bank,
the same distance, and the same scoring rule; only the representation differs. The gap
between them is what the features are worth on this task.

It does **not** show that training is unnecessary in general. Published methods that do
train, and methods that subsample the bank properly with a coreset, score higher than a
plain 1-NN over a random subsample. And image-level AUROC says nothing about whether the
defect is *localised* correctly - that needs the pixel-level masks, which this notebook
does not score.

Part of a portfolio: [github.com/hammasbuilds](https://github.com/hammasbuilds) ·
[vision-portfolio-woad.vercel.app](https://vision-portfolio-woad.vercel.app)
